In [23]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

In [24]:
df = pd.read_csv('data/adult.csv', index_col=False) # csv 파일 헤더 없을 때 내가 넣어야 하고 헤더는 띄어쓰기 하지 않는 것이 규약 !!! 데이터는 꼭 정제가 잘 된 것을 쓰자
# df.to_csv('temp.csv', index=False) # index=False : csv 저장 시 앞에 자동으로 들어가는 인덱스 없애줌

## Q. income을 예측하는 간단한 모델을 작성해보세요.

- 이진분류 문제
    - 답안이 income(2개로 구성)

In [25]:
df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


1. income 내용 확인

In [26]:
df[['income']].value_counts() # 비율 안 맞으니 섞을 때 조심히 섞어야 함 //  두 개니까 이진분류 해야 함(?)

income
<=50K     24720
>50K       7841
Name: count, dtype: int64

2. 현재 데이터 어떤지 확인

In [27]:
df.isnull().sum() # 처리할 게 없네네

age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
gender            0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
income            0
dtype: int64

## Q1. X, y(학습데이터와 정답지를 나눔)

In [28]:
X = df.drop("income", axis=1)
y = df["income"].apply(lambda x: 1 if x.strip() == "<=50K" else 0) # "<=50K" is 1, ">50K" is 0
y.value_counts()

income
1    24720
0     7841
Name: count, dtype: int64

## Q2. 데이터를 split하세요.

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=y)

- 판다스는 정형데이터를 다루는 도구 (like SQL, Excel) 다른 점 : 자동화
1. 데이터 불러오기 / 내보내기
2. 수학함수(f(x)) <- numpy, scipy
3. 제어문: if, 필터링
    반복문: for(loc, iloc) -> apply(f(x)) 대부분 람다를 씀 무조건 return하니까 // 속도 빠르고 아주 중요

## Q3. 수치형 데이터와 범주형 데이터를 분류

In [30]:
numeric_feat = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
categorical_feat = ["workclass", "education", "marital-status", "occupation", "relationship", "race", "gender", "native-country"] # column만 분리

In [31]:
# 수치형
numeric_transformer = SimpleImputer(strategy="median") # 중위값으로 빈 데이터에 밀어넣음
X_train_numeric = numeric_transformer.fit_transform(X_train[numeric_feat]) # numeric_feat가 하나씩 들어가면서
X_test_numeric = numeric_transformer.transform(X_test[numeric_feat]) # fit 안 함 : 학습한 모델을 기준으로 바꾸는 거임

In [32]:
# 수치형
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_numeric)
X_test_scaled = scaler.transform(X_test_numeric)

In [33]:
# 범주형
categorical_transformer = SimpleImputer(strategy="constant", fill_value="missing") # 파라미터 이게 짝
X_train_categorical = categorical_transformer.fit_transform(X_train[categorical_feat])
X_test_categorical = categorical_transformer.transform(X_test[categorical_feat])

In [34]:
# 범주형
encoder = OneHotEncoder(handle_unknown="ignore") # 플라스크에서 로그로 저장해둬야 함
X_train_encoded = encoder.fit_transform(X_train_categorical)
X_test_encoded = encoder.transform(X_test_categorical)

합치려면?
(X_train_encoded, X_train_scaled), (X_test_encoded, X_test_scaled)
1. numpy
2. 어느방향으로?

In [35]:
X_train_processed = np.hstack((X_train_scaled, X_train_encoded.toarray()))
X_test_processed = np.hstack((X_test_scaled, X_test_encoded.toarray()))

In [36]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)

RandomForestClassifier(random_state=42)

In [37]:
y_pred = model.predict(X_test_processed)
accuracy_score(y_test, y_pred)

0.8485444048642673

- feature 이름을 같이 뽑음

In [38]:
feat_names = (numeric_feat + encoder.get_feature_names_out(categorical_feat).tolist()) # 뭐가 중요하니?
feat_imp = model.feature_importances_
sorted_idx = np.argsort(feat_imp)[::-1]
for i in sorted_idx[:10]:
    if i < len(feat_names):
        print(f'{feat_names[i]}: {feat_imp[i]}')

fnlwgt: 0.15689703480344602
age: 0.14950178726392593
capital-gain: 0.09092639056770868
hours-per-week: 0.08232124210935983
marital-status_ Married-civ-spouse: 0.06884090551932814
education-num: 0.060008472977840156
relationship_ Husband: 0.037862151852119864
capital-loss: 0.029823796109959395
marital-status_ Never-married: 0.028194891743726667
occupation_ Exec-managerial: 0.019629802364583362


## Q6.  

테이블 복사(select 권한 받으면)
- select * into [newtablename] from [originaltablename];
이건 내꺼, 컬럼명 예쁘게 만들자

In [39]:
#  age, workclass, education, gender, hours-per-week, occupation, income
# data['age', 'workclass'] => 오류, 인덱스가 두 개인 것은 안 됨 그래서 리스트로 복합키를 만들어줌? // data[['age']] 하나를 쓰더라도 이렇게 써서 헷갈리지 않게 함
# data = df[['age', 'workclass', 'education', 'gender', 'hours-per-week', 'occupation', 'income']]

- 배열 : 모노 타입으로 크기가 고정된 연속된 구조
- list가 배열의 특성을 가져옴 => 인덱스 !!

In [40]:
# data[['gender']].value_counts()

In [41]:
# pd.get_dummies(data) # 원핫인코딩(희소행렬) => 장점 : 잘 됨, 읽기 좋음